### 策略名称: 多因子复合微观结构因子 (Multi-Factor Composite Microstructure Alpha)

**策略概述:**
本策略融合四个维度的日内微观结构信号，构建15分钟频率的复合因子：

1. **订单簿不平衡因子 (Order Book Imbalance):** 利用5档盘口加权买卖量差异捕捉市场供需关系。
2. **日内动量反转因子 (Intraday Reversal):** 收盘价相对窗口均价的偏离度取反，利用均值回归特性。
3. **成交量加速度因子 (Volume Acceleration):** 后半窗口vs前半窗口的成交量比率，捕捉资金流入加速。
4. **大单净流入因子 (Net Order Flow):** 利用盘口深度的变化估算机构资金方向。

---

**数学逻辑 (Mathematical Logic):**

1.  **订单簿不平衡 (OBI):**
    $$W_{bid} = \sum_{i=1}^{5} BidVol_i \times e^{-0.3(i-1)}, \quad W_{ask} = \sum_{i=1}^{5} AskVol_i \times e^{-0.3(i-1)}$$
    $$OBI = \frac{W_{bid} - W_{ask}}{W_{bid} + W_{ask} + \epsilon}$$

2.  **日内反转 (Reversal):**
    $$Rev = -1 \times \tanh\left(\frac{P_{close} - P_{avg}}{P_{avg}} \times 10\right)$$

3.  **成交量加速度 (Volume Acceleration):**
    窗口内后半段vs前半段成交量比率的对数变换。

4.  **净流入 (Net Flow):**
    $$NetFlow = \frac{TotalBid_{3} - TotalAsk_{3}}{TotalBid_{3} + TotalAsk_{3} + \epsilon}$$

5.  **复合因子:**
    $$Factor = \tanh\left(0.30 \times Z(OBI) + 0.35 \times Z(Rev) + 0.15 \times Z(VolAccel) + 0.20 \times Z(NetFlow)\right)$$

---

**设计思路:**
- 订单簿因子捕捉盘口微观信号（supply/demand），日内反转捕捉均值回归
- 成交量加速度衡量资金流入的动态变化，净流入衡量机构方向
- 多维信号融合降低单因子失效风险，tanh缩放控制极端值
- 权重设定：反转信号(0.35)为主，盘口信号(0.30+0.20)为辅，量能加速(0.15)为补充


In [ ]:
def main(datasource, start_date, end_date):
    """
    Multi-Factor Composite Microstructure Alpha
    融合订单簿不平衡、日内反转、成交量加速度、净流入四个信号

    Args:
        datasource (str): Datasource table name
        start_date (str): Start date in 'YYYY-MM-DD HH:MM:SS' format
        end_date (str): End date in 'YYYY-MM-DD HH:MM:SS' format

    Returns:
        pd.DataFrame: Factor data with columns ['date', 'instrument', 'factor']
    """
    import pandas as pd
    import dai

    # 复合因子权重
    w_obi = 0.30       # 订单簿不平衡
    w_rev = 0.35       # 日内反转
    w_vol_accel = 0.15 # 成交量加速度
    w_net_flow = 0.20  # 净流入

    sql = f"""
    -- 优化设置
    SET preserve_insertion_order=false;
    SET threads=4;

    WITH cte_snapshot AS (
        SELECT
            date,
            instrument_id,

            -- 中间价
            (ask_price1 + bid_price1) / 2.0 AS mid_price,

            -- 成交量
            volume,

            -- 加权5档买方量 (指数衰减)
            (
                COALESCE(bid_volume1, 0) * 1.0 +
                COALESCE(bid_volume2, 0) * EXP(-0.3) +
                COALESCE(bid_volume3, 0) * EXP(-0.6) +
                COALESCE(bid_volume4, 0) * EXP(-0.9) +
                COALESCE(bid_volume5, 0) * EXP(-1.2)
            ) AS weight_bid,

            -- 加权5档卖方量 (指数衰减)
            (
                COALESCE(ask_volume1, 0) * 1.0 +
                COALESCE(ask_volume2, 0) * EXP(-0.3) +
                COALESCE(ask_volume3, 0) * EXP(-0.6) +
                COALESCE(ask_volume4, 0) * EXP(-0.9) +
                COALESCE(ask_volume5, 0) * EXP(-1.2)
            ) AS weight_ask,

            -- 订单簿不平衡度 (OBI)
            (weight_bid - weight_ask) / (weight_bid + weight_ask + 1e-8) AS obi,

            -- 3档净深度 (Net Flow)
            (
                COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0)
            ) AS total_bid_3,
            (
                COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0)
            ) AS total_ask_3,
            (total_bid_3 - total_ask_3) / (total_bid_3 + total_ask_3 + 1e-8) AS net_flow,

            -- 交易日
            strftime(date, '%Y-%m-%d') AS trading_day,

            -- 15分钟分段
            CASE
                WHEN strftime(date, '%H%M') >= '0930' AND strftime(date, '%H%M') < '0945' THEN 94500
                WHEN strftime(date, '%H%M') >= '0945' AND strftime(date, '%H%M') < '1000' THEN 100000
                WHEN strftime(date, '%H%M') >= '1000' AND strftime(date, '%H%M') < '1015' THEN 101500
                WHEN strftime(date, '%H%M') >= '1015' AND strftime(date, '%H%M') < '1030' THEN 103000
                WHEN strftime(date, '%H%M') >= '1030' AND strftime(date, '%H%M') < '1045' THEN 104500
                WHEN strftime(date, '%H%M') >= '1045' AND strftime(date, '%H%M') < '1100' THEN 110000
                WHEN strftime(date, '%H%M') >= '1100' AND strftime(date, '%H%M') < '1115' THEN 111500
                WHEN strftime(date, '%H%M') >= '1115' AND strftime(date, '%H%M') <= '1130' THEN 113000
                WHEN strftime(date, '%H%M') >= '1300' AND strftime(date, '%H%M') < '1315' THEN 131500
                WHEN strftime(date, '%H%M') >= '1315' AND strftime(date, '%H%M') < '1330' THEN 133000
                WHEN strftime(date, '%H%M') >= '1330' AND strftime(date, '%H%M') < '1345' THEN 134500
                WHEN strftime(date, '%H%M') >= '1345' AND strftime(date, '%H%M') < '1400' THEN 140000
                WHEN strftime(date, '%H%M') >= '1400' AND strftime(date, '%H%M') < '1415' THEN 141500
                WHEN strftime(date, '%H%M') >= '1415' AND strftime(date, '%H%M') < '1430' THEN 143000
                WHEN strftime(date, '%H%M') >= '1430' AND strftime(date, '%H%M') < '1445' THEN 144500
                WHEN strftime(date, '%H%M') >= '1445' AND strftime(date, '%H%M') < '1457' THEN 150000
                ELSE -1
            END AS time_segment
        FROM {datasource}
    ),

    cte_filtered AS (
        SELECT *
        FROM cte_snapshot
        WHERE time_segment != -1
          AND mid_price IS NOT NULL
          AND mid_price > 0
    ),

    -- 窗口内按时间排序，标记行号用于成交量加速度
    cte_ordered AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY instrument_id, trading_day, time_segment
                ORDER BY date
            ) AS rn,
            COUNT(*) OVER (
                PARTITION BY instrument_id, trading_day, time_segment
            ) AS total_cnt
        FROM cte_filtered
    ),

    -- 15分钟窗口聚合
    cte_window AS (
        SELECT
            instrument_id,
            trading_day,
            time_segment,

            -- 子因子1: 订单簿不平衡 (OBI) - 窗口均值
            AVG(obi) AS obi_mean,

            -- 子因子2: 日内反转 - 用close vs avg
            argMax(mid_price, date) AS close_mid,
            AVG(mid_price) AS avg_mid,

            -- 子因子3: 成交量加速度 - 后半段vs前半段成交量
            SUM(CASE WHEN rn > total_cnt / 2 THEN volume ELSE 0 END) AS vol_second_half,
            SUM(CASE WHEN rn <= total_cnt / 2 THEN volume ELSE 0 END) AS vol_first_half,

            -- 子因子4: 净流入 - 窗口均值
            AVG(net_flow) AS net_flow_mean

        FROM cte_ordered
        GROUP BY instrument_id, trading_day, time_segment
    ),

    -- 截面标准化 (在每个 trading_day + time_segment 截面内 Z-Score)
    cte_cross_section AS (
        SELECT
            instrument_id,
            trading_day,
            time_segment,

            -- OBI Z-Score (截面标准化)
            (obi_mean - AVG(obi_mean) OVER (PARTITION BY trading_day, time_segment))
            / (nanstd(obi_mean) OVER (PARTITION BY trading_day, time_segment) + 1e-8)
            AS obi_zscore,

            -- 日内反转因子 Z-Score (截面标准化)
            -1.0 * tanh(((close_mid - avg_mid) / (avg_mid + 1e-8)) * 10.0) AS rev_raw,
            (rev_raw - AVG(rev_raw) OVER (PARTITION BY trading_day, time_segment))
            / (nanstd(rev_raw) OVER (PARTITION BY trading_day, time_segment) + 1e-8)
            AS rev_zscore,

            -- 成交量加速度 Z-Score
            CASE
                WHEN vol_first_half > 0
                THEN LN((vol_second_half + 1e-8) / (vol_first_half + 1e-8))
                ELSE 0
            END AS vol_accel_raw,
            (vol_accel_raw - AVG(vol_accel_raw) OVER (PARTITION BY trading_day, time_segment))
            / (nanstd(vol_accel_raw) OVER (PARTITION BY trading_day, time_segment) + 1e-8)
            AS vol_accel_zscore,

            -- 净流入 Z-Score
            (net_flow_mean - AVG(net_flow_mean) OVER (PARTITION BY trading_day, time_segment))
            / (nanstd(net_flow_mean) OVER (PARTITION BY trading_day, time_segment) + 1e-8)
            AS net_flow_zscore

        FROM cte_window
    ),

    -- 复合因子
    cte_composite AS (
        SELECT
            instrument_id,
            trading_day,
            time_segment,
            tanh(
                {w_obi} * obi_zscore
                + {w_rev} * rev_zscore
                + {w_vol_accel} * vol_accel_zscore
                + {w_net_flow} * net_flow_zscore
            ) AS factor
        FROM cte_cross_section
    )

    -- 映射 instrument_id 到 instrument 列
    SELECT
        CAST(CONCAT(
            c.trading_day, ' ',
            strftime(strptime(LPAD(c.time_segment, 6, '0'), '%H%M%S'), '%H:%M:%S')
        ) AS DATETIME) AS date,
        all_instruments.instrument,
        c.factor
    FROM cte_composite c
    LEFT JOIN all_instruments USING (instrument_id)
    """

    df = dai.query(sql, filters={"date": [start_date, end_date]}).df()
    return df


if __name__ == "__main__":
    """
    开发调试专用模块：分块循环回测引擎
    """
    from bigmodule import M
    import pandas as pd
    import structlog
    import gc
    from concurrent.futures import ThreadPoolExecutor, as_completed

    logger = structlog.get_logger()
    datasource = "cpt_dwc_2026_stock_hs300_snapshot"

    full_start_date = "2023-01-01"
    full_end_date = "2024-12-01"

    # 准备时间分片列表
    date_ranges = pd.date_range(start=full_start_date, end=full_end_date, freq="MS")
    chunk_params = []
    for start_dt in date_ranges:
        current_start = start_dt.strftime("%Y-%m-%d 00:00:00")
        current_end = (start_dt + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d 23:59:59")
        chunk_params.append((current_start, current_end))

    all_results = []
    max_workers = 4

    logger.info(f"🚀 Starting Multi-Factor Composite Backtest: {full_start_date} to {full_end_date}")
    logger.info(f"⚡ Parallel Workers: {max_workers}")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_date = {
            executor.submit(main, datasource, start, end): (start, end)
            for start, end in chunk_params
        }

        for future in as_completed(future_to_date):
            start, end = future_to_date[future]
            try:
                df_chunk = future.result()
                if df_chunk is not None and not df_chunk.empty:
                    all_results.append(df_chunk)
                    logger.info(f"✅ Chunk Done: {start[:7]} | Rows: {len(df_chunk)}")
                else:
                    logger.warning(f"⚠️ Chunk Empty: {start[:7]}")
                del df_chunk
                gc.collect()
            except Exception as e:
                logger.error(f"❌ Error in chunk {start}: {e}")

    if all_results:
        logger.info("🧩 Concatenating all chunks...")
        final_data = pd.concat(all_results, ignore_index=True)
        final_data.sort_values(by=["date", "instrument"], inplace=True)

        logger.info(f"🎉 All Done! Final Shape: {final_data.shape}")
        logger.info(f"Sample:\n{final_data.head()}")

        logger.info("📊 Starting Evaluation...")
        try:
            _ = M.eval_dwc._latest(data=final_data)
        except Exception as e:
            logger.warning(f"Evaluation failed: {e}")
            print("Data preview:", final_data.head())
    else:
        logger.error("No data generated.")
